# 1

In [2]:
import os
import logging
import asyncio
from datetime import datetime, timezone, timedelta

from dotenv import load_dotenv
from rich.logging import RichHandler

import yfinance as yf
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    OpenAIChatCompletionsModel,
    Runner,
    function_tool,
    set_tracing_disabled,
)

# ----- Logging bonito en consola -----
logging.basicConfig(level=logging.INFO, format="%(message)s", handlers=[RichHandler()])

# Cargar variables del .env
load_dotenv()

# Desactivar tracing si no lo usas
set_tracing_disabled(True)

# ----- Azure OpenAI Configuration -----
endpoint = os.getenv("ENDPOINT_URL")
deployment = os.getenv("DEPLOYMENT_NAME")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("OPENAI_API_VERSION", "2025-01-01-preview")

if not all([endpoint, deployment, subscription_key]):
    raise RuntimeError(
        "Faltan variables de entorno: ENDPOINT_URL, DEPLOYMENT_NAME o AZURE_OPENAI_API_KEY."
    )

client = AsyncAzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

MODEL_NAME = deployment


# 2

In [6]:
def get_current_time() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d")

@function_tool
def get_current_date() -> str:
    return get_current_time()

In [3]:
import os
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

# Los permisos que el agente necesita (leer y escribir eventos)
SCOPES = ['https://www.googleapis.com/auth/calendar.events']

def authenticate_google_calendar():
    creds = None
    
    # Verifica si ya existe el token.json de una sesión anterior
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    # Si no hay credenciales válidas o expiraron, te pedirá iniciar sesión
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # AQUÍ es donde tu código lee el archivo credentials.json que acabas de descargar
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            
            # Esto abrirá una pestaña en tu navegador web
            creds = flow.run_local_server(port=0)
        
        # Guarda el token generado para que el agente lo use automáticamente después
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
            
    print("¡Autenticación exitosa! El archivo token.json se ha creado correctamente.")
    return creds

# Ejecutamos la función de autenticación
authenticate_google_calendar()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1081828649459-1o19ror126r669iudbmdl6u62lcagd8u.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A64770%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.events&state=rKBmrAaSoyNnAbjw6vxLLdOcXNqa23&access_type=offline


[02/25/26 17:33:31] INFO     "GET                                                                       flow.py:476
                             /?state=rKBmrAaSoyNnAbjw6vxLLdOcXNqa23&iss=https://accounts.google.com&cod            
                             e=4/0AfrIepC2XTPSWn7pctHHlFef2FVdQGzqA8kr3kJM4EEiN5Ll_f36-Lk1PxpyrtLJRgQUc            
                             A&scope=https://www.googleapis.com/auth/calendar.events HTTP/1.1" 200 65              

¡Autenticación exitosa! El archivo token.json se ha creado correctamente.


In [4]:
import os
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials

@function_tool
def create_calendar_event(summary: str, start_time: str, end_time: str, description: str = "") -> str:
    """
    Crea un nuevo evento en el Google Calendar principal del usuario.
    
    Args:
        summary: El título o resumen del evento.
        start_time: Fecha y hora de inicio en formato ISO 8601 (ej. '2026-02-26T10:00:00-04:00').
        end_time: Fecha y hora de finalización en formato ISO 8601 (ej. '2026-02-26T11:00:00-04:00').
        description: Detalles opcionales del evento.
    """
    SCOPES = ['https://www.googleapis.com/auth/calendar.events']
    
    # 1. Verificar que el token exista (generado en el paso anterior)
    if not os.path.exists('token.json'):
        return "Error: No se encontró 'token.json'. Por favor, ejecuta la autenticación primero."
        
    try:
        # 2. Cargar las credenciales
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
        
        # 3. Construir el cliente de la API de Google Calendar
        service = build('calendar', 'v3', credentials=creds)

        # 4. Estructurar el evento
        event = {
            'summary': summary,
            'description': description,
            'start': {
                'dateTime': start_time,
                'timeZone': 'America/La_Paz', # Zona horaria ajustada a tu ubicación
            },
            'end': {
                'dateTime': end_time,
                'timeZone': 'America/La_Paz',
            },
        }

        # 5. Insertar el evento en el calendario principal
        created_event = service.events().insert(calendarId='primary', body=event).execute()
        
        # Devolvemos un mensaje de éxito con el enlace para que el Agente se lo muestre al usuario
        return f"Éxito: Evento '{summary}' creado correctamente. Enlace: {created_event.get('htmlLink')}"
        
    except Exception as e:
        # Es importante devolver los errores como texto para que el agente sepa qué falló
        return f"Error de la API al crear el evento: {str(e)}"

# Creando el Agent

In [7]:
calendar_assistant= Agent(
    name="Calendar Assistant",
    instructions=(
    "Eres un asistente personal experto en gestión de tiempo, productividad y organización de agendas. "
    "Tu objetivo principal es ayudar al usuario a gestionar su Google Calendar, creando eventos, tareas y planificando fechas. "
    "Reglas estrictas que debes seguir:\n"
    "1. CONTEXTO TEMPORAL: Antes de responder a cualquier petición sobre fechas o programar un evento, siempre debes usar la herramienta para obtener la fecha y hora actual exacta. "
    "2. PREVENCIÓN DE ERRORES: Si el usuario te pide agendar algo (ej. 'reunión mañana'), pero no especifica la duración o la hora exacta de inicio/fin, DEBES preguntarle esos detalles antes de intentar crear el evento. "
    "3. ZONA HORARIA: Asume siempre que la zona horaria del usuario es America/La_Paz (Bolivia, UTC-4) a menos que se especifique lo contrario. "
    "4. CONFIRMACIÓN: Una vez que ejecutes la herramienta de creación de eventos con éxito, responde de manera concisa confirmando la acción y proporcionando el enlace al evento."
    ),
    tools=[get_current_date, create_calendar_event],
    model=OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=client),
)

In [9]:
async def demo():
    # Un prompt de prueba realista para tu asistente
    prompt = "Agéndame una reunión de revisión de código para hoy a las 17:53. Que dure exactamente una hora."
    
    # Ejecutamos el agente pasándole tu calendar_assistant
    result = await Runner.run(calendar_assistant, input=prompt)
    
    # Imprimimos la respuesta final
    print(result.final_output)

await demo()

[02/25/26 17:53:02] INFO     HTTP Request: POST                                                     _client.py:1740
                             https://khipusaigpt0566189501.openai.azure.com/openai/deployments/gpt-                
                             4o/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"                  

[02/25/26 17:53:03] INFO     HTTP Request: POST                                                     _client.py:1740
                             https://khipusaigpt0566189501.openai.azure.com/openai/deployments/gpt-                
                             4o/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"                  

                    INFO     file_cache is only supported with oauth2client<4.0.0                    __init__.py:49

[02/25/26 17:53:06] INFO     HTTP Request: POST                                                     _client.py:1740
                             https://khipusaigpt0566189501.openai.azure.com/openai/deployments/gpt-                
                             4o/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"                  

La reunión de revisión de código ha sido programada para hoy, 25 de febrero de 2026, de 17:53 a 18:53 (UTC-4). Puedes encontrar el evento [aquí](https://www.google.com/calendar/event?eid=dDNzcXIzaW01cTg0Zzl1cGZwMHBmdWF2YmcgcXVpc3BlanAxOUBt).
